In [7]:
import torch
import os 
import json
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import CIFAR100
from torchvision import transforms, models
%cd /home/devops/data/DL4CV/DL4CV/A1/Retrain
# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# TIỀN XỬ LÝ DỮ LIỆU & AUGMENTATION
# ==========================================
transform_train = transforms.Compose([
    transforms.Resize(224),
    transforms.RandomCrop(224, padding=4),
    transforms.RandomHorizontalFlip(),

    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),

    transforms.ToTensor(),

    transforms.Normalize(
        (0.485, 0.456, 0.406),
        (0.229, 0.224, 0.225)
    ),

    transforms.RandomErasing(
        p=0.25,
        scale=(0.02, 0.15),
        ratio=(0.3, 3.3)
    )
])
# Chuẩn hóa theo Mean và Std của bộ dữ liệu CIFAR-100
# transform_train = transforms.Compose([
#     #transforms.Resize(224),  
#     transforms.RandomCrop(32, padding=4),
#     transforms.RandomHorizontalFlip(),
#     transforms.ToTensor(),
#     transforms.Normalize((0.5071, 0.4867, 0.4408), (0.2675, 0.2565, 0.2761))
# ])

transform_test = transforms.Compose([
    transforms.Resize(224),
    transforms.ToTensor(),
    transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
])
# 4. Khởi tạo Dataset chuẩn
raw_train_dataset  = CIFAR100(root="data", train=True, download=True, transform=transform_train)
raw_val_dataset    = CIFAR100(root="data", train=True, download=False, transform=transform_test)
final_test_dataset = CIFAR100(root="data", train=False, download=True, transform=transform_test)

# 5. Chia Train (90%) - Val (10%) từ 50,000 mẫu
total_samples = len(raw_train_dataset)
train_size = int(0.9 * total_samples)
val_size = total_samples - train_size

generator = torch.Generator().manual_seed(42)
permuted_indices = torch.randperm(total_samples, generator=generator).tolist()

train_indices = permuted_indices[:train_size]
val_indices   = permuted_indices[train_size:]

TRAIN_SUBSET = Subset(raw_train_dataset, train_indices)
VAL_SUBSET   = Subset(raw_val_dataset, val_indices)

# 6. DataLoader với num_workers=0 để tránh lỗi Shared Memory (shm)
BATCH_SIZE = 64
train_loader = DataLoader(TRAIN_SUBSET, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
val_loader   = DataLoader(VAL_SUBSET, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)
test_loader  = DataLoader(final_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0, pin_memory=True)

/home/devops/data/DL4CV/DL4CV/A1/Retrain


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load ResNet18 pretrained on ImageNet-1K
weights = models.ResNet18_Weights.DEFAULT

model = models.resnet18(weights=weights)

# Replace the original classifier (1000 classes) with CIFAR-100 classifier
model.fc = nn.Linear(model.fc.in_features, 100)

# Unfreeze all layers for fine-tuning
for param in model.parameters():
    param.requires_grad = True

model = model.to(device)

In [9]:
optimizer = optim.AdamW(
    model.parameters(),
    lr=0.0001,
    weight_decay=0.01
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=50,
    eta_min=1e-6
)

criterion = nn.CrossEntropyLoss()

In [10]:
def train_one_epoch_finetune(model, train_loader, criterion, optimizer, device):
    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        # Clear gradients
        optimizer.zero_grad()

        # Forward
        outputs = model(images)
        loss = criterion(outputs, labels)

        # Backward
        loss.backward()

        # Update all trainable parameters
        optimizer.step()

        running_loss += loss.item() * images.size(0)

        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100.0 * correct / total

    return epoch_loss, epoch_accuracy

In [11]:
def evaluate(model, val_loader, criterion, device):
    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)

            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / total
    epoch_accuracy = 100.0 * correct / total

    return epoch_loss, epoch_accuracy

In [13]:
import time
from experiment_logger import ExperimentLogger
SEED = 42
NUM_EPOCHS = 50
BATCH_SIZE = 64

torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

config = {
    "optimizer": "AdamW",
    "learning_rate": 0.0001,
    "weight_decay": 0.01,
    "scheduler": "CosineAnnealingLR",
    "scheduler_T_max": 50,
    "scheduler_eta_min": 1e-6,
    "criterion": "CrossEntropyLoss",
    "pretrained": True,
    "pretrained_dataset": "ImageNet-1K",
    "fine_tuning": True,
    "frozen_backbone": False,
    "input_size": [224, 224],
    "num_classes": 100,
    "epochs": NUM_EPOCHS,
    "batch_size": BATCH_SIZE
}
os.makedirs("results/checkpoints", exist_ok=True)

checkpoint_path = (
    "results/checkpoints/"
    "best_resnet18_finetuning.pth"
)

# ============================================================
# 5. Initialize Experiment Logger
# ============================================================

best_val_acc = 0.0
start_time = time.time()

logger = ExperimentLogger(
    experiment_id="ResNet18_FineTuning_seed42",
    model_name="ResNet18",
    strategy="Pretrained_FineTuning",
    seed=SEED,
    input_size=[224, 224],
    config=config
)

# Log model complexity
logger.log_model_complexity(model)

# Reset GPU peak memory statistics
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats(device)

# ============================================================
# 6. Training
# ============================================================

try:
    for epoch in range(NUM_EPOCHS):

        epoch_start = time.time()

        # ----------------------------------------------------
        # Training: update the entire ResNet18 network
        # ----------------------------------------------------

        train_loss, train_acc = train_one_epoch_finetune(
            model,
            train_loader,
            criterion,
            optimizer,
            device=device
        )

        # ----------------------------------------------------
        # Validation
        # ----------------------------------------------------

        val_loss, val_acc = evaluate(
            model,
            val_loader,
            criterion,
            device=device
        )

        # Update learning rate
        scheduler.step()

        epoch_time = time.time() - epoch_start

        # ----------------------------------------------------
        # Collect GPU memory information
        # ----------------------------------------------------

        if torch.cuda.is_available():
            memory_allocated = (
                torch.cuda.memory_allocated(device) / 1024**2
            )
            memory_reserved = (
                torch.cuda.memory_reserved(device) / 1024**2
            )
            peak_memory_allocated = (
                torch.cuda.max_memory_allocated(device) / 1024**2
            )
            peak_memory_reserved = (
                torch.cuda.max_memory_reserved(device) / 1024**2
            )
        else:
            memory_allocated = 0.0
            memory_reserved = 0.0
            peak_memory_allocated = 0.0
            peak_memory_reserved = 0.0

        # ----------------------------------------------------
        # Log epoch
        # ----------------------------------------------------

        logger.log_epoch(epoch + 1, {
            "epoch": epoch + 1,
            "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
            "train_loss": train_loss,
            "train_accuracy": train_acc,
            "val_loss": val_loss,
            "val_accuracy": val_acc,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "epoch_time_sec": epoch_time,
            "samples_per_sec": (
                len(train_loader.dataset) / epoch_time
            ),
            "memory_allocated_mb": memory_allocated,
            "memory_reserved_mb": memory_reserved,
            "peak_memory_allocated_mb": peak_memory_allocated,
            "peak_memory_reserved_mb": peak_memory_reserved
        })

        # ----------------------------------------------------
        # Print progress
        # ----------------------------------------------------

        print(
            f"Epoch [{epoch + 1}/{NUM_EPOCHS}] "
            f"Train Loss: {train_loss:.4f} | "
            f"Train Acc: {train_acc:.2f}% | "
            f"Val Loss: {val_loss:.4f} | "
            f"Val Acc: {val_acc:.2f}% | "
            f"LR: {optimizer.param_groups[0]['lr']:.7f}"
        )

        # ----------------------------------------------------
        # Save best model based on validation accuracy
        # ----------------------------------------------------

        if val_acc > best_val_acc:

            best_val_acc = val_acc

            torch.save({
                "epoch": epoch + 1,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "scheduler_state_dict": scheduler.state_dict(),
                "val_acc": val_acc,
                "val_loss": val_loss
            }, checkpoint_path)

            logger.log_best_checkpoint(
                epoch + 1,
                val_acc
            )

    # ========================================================
    # 7. Load Best Checkpoint
    # ========================================================

    checkpoint = torch.load(
        checkpoint_path,
        map_location=device,
        weights_only=True
    )

    model.load_state_dict(checkpoint["model_state_dict"])

    print(
        f"\nLoaded best checkpoint from epoch "
        f"{checkpoint['epoch']}"
    )

    # ========================================================
    # 8. Final Test Evaluation
    # ========================================================

    test_loss, test_acc = evaluate(
        model,
        test_loader,
        criterion,
        device
    )

    training_time = time.time() - start_time

    # ========================================================
    # 9. Log Test Results
    # ========================================================

    logger.log_test_results({
        "test_loss": test_loss,
        "test_accuracy": test_acc
    })

    # ========================================================
    # 10. Finish Experiment
    # ========================================================

    logger.finish(
        status="completed",
        extra_metrics={
            "best_val_accuracy": best_val_acc,
            "training_time_sec": training_time,
            "test_loss": test_loss,
            "test_accuracy": test_acc
        }
    )

    print("\n========== EXPERIMENT RESULTS ==========")
    print(f"Best Validation Accuracy: {best_val_acc:.2f}%")
    print(f"Test Loss: {test_loss:.4f}")
    print(f"Test Accuracy: {test_acc:.2f}%")
    print(f"Training Time: {training_time:.2f}s")
    print("========================================")

except Exception as e:

    logger.log_event("experiment_failed", {
        "error": str(e)
    })

    logger.finish(status="failed")

    raise

Epoch [1/50] Train Loss: 1.8444 | Train Acc: 56.15% | Val Loss: 0.9981 | Val Acc: 71.90% | LR: 0.0000999
Epoch [2/50] Train Loss: 0.8828 | Train Acc: 75.68% | Val Loss: 0.7690 | Val Acc: 76.90% | LR: 0.0000996
Epoch [3/50] Train Loss: 0.6318 | Train Acc: 82.07% | Val Loss: 0.7259 | Val Acc: 78.20% | LR: 0.0000991
Epoch [4/50] Train Loss: 0.4918 | Train Acc: 85.98% | Val Loss: 0.6842 | Val Acc: 79.22% | LR: 0.0000984
Epoch [5/50] Train Loss: 0.3772 | Train Acc: 89.23% | Val Loss: 0.6692 | Val Acc: 80.02% | LR: 0.0000976
Epoch [6/50] Train Loss: 0.2922 | Train Acc: 91.80% | Val Loss: 0.6813 | Val Acc: 80.36% | LR: 0.0000965
Epoch [7/50] Train Loss: 0.2356 | Train Acc: 93.45% | Val Loss: 0.6967 | Val Acc: 79.64% | LR: 0.0000953
Epoch [8/50] Train Loss: 0.1935 | Train Acc: 94.64% | Val Loss: 0.7090 | Val Acc: 80.02% | LR: 0.0000939
Epoch [9/50] Train Loss: 0.1633 | Train Acc: 95.50% | Val Loss: 0.6918 | Val Acc: 80.48% | LR: 0.0000923
Epoch [10/50] Train Loss: 0.1359 | Train Acc: 96.36% | 

In [16]:
!python plot_experiment_results.py

Skip failed experiment: ResNet18_Pretrained_Finetune_seed42
Tìm thấy 3 experiment hợp lệ.
Created learning curves: ResNet18_Scratch_seed42
Created learning curves: ResNet18_Pretrained_FineTuning_seed42
Created learning curves: ResNet18_FineTuning_seed42
Created comparison: test_accuracy.png
Created comparison: best_val_accuracy.png
Created comparison: training_time.png
Created comparison: gpu_memory.png
Created comparison: model_parameters.png
Created comparison: trainable_parameters.png

========== EXPERIMENT SUMMARY ==========

Experiment: ResNet18_Scratch_seed42
Best Epoch: 47
Best Val Accuracy: 66.96
Test Accuracy: 66.8
Training Time: 1846.45 sec
Peak GPU Memory: 824.83 MB
Total Parameters: 11220132
Trainable Parameters: 11220132

Experiment: ResNet18_Pretrained_FineTuning_seed42
Best Epoch: 38
Best Val Accuracy: 60.5
Test Accuracy: None
Training Time: 5557.776336669922 sec
Peak GPU Memory: 572.92 MB
Total Parameters: 11227812
Trainable Parameters: 51300

Experiment: ResNet18_FineT

In [15]:
import json

json_path = "results/experiment_results.json"

with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

experiments = data.get("experiments", [])

print("Total experiments:", len(experiments))

for exp in experiments:

    print("\n================================")

    print("Experiment ID:",
          exp.get("experiment_id"))

    print("Metadata:",
          exp.get("metadata"))

    print("Available keys:",
          list(exp.keys()))

    history = exp.get("training_history", [])

    print("Training history length:", len(history))

    if history:
        print("First epoch:", history[0])
        print("Last epoch:", history[-1])

    print("Best checkpoint:",
          exp.get("best_checkpoint"))

    print("Testing:",
          exp.get("testing"))

    print("Computation:",
          exp.get("computation"))

    print("Complexity:",
          exp.get("complexity"))

Total experiments: 4

Experiment ID: ResNet18_Scratch_seed42
Metadata: {'model': 'ResNet18', 'strategy': 'Scratch', 'seed': 42, 'input_size': [32, 32], 'start_time': '2026-10-31T13:51:40.172281', 'end_time': '2026-10-31T14:22:29.011058', 'status': 'completed'}
Available keys: ['experiment_id', 'metadata', 'configuration', 'training_history', 'best_checkpoint', 'testing', 'computation', 'events', 'complexity']
Training history length: 50
First epoch: {'epoch': 1, 'timestamp': '2026-10-31T13:52:34.313627', 'train_loss': 3.8093, 'train_accuracy': 11.95, 'val_loss': 3.3966, 'val_accuracy': 18.36, 'learning_rate': 0.0001, 'epoch_time_sec': 54.139, 'samples_per_sec': 863.44, 'memory_allocated_mb': 544.12, 'memory_reserved_mb': 916.0, 'peak_memory_allocated_mb': 824.83, 'peak_memory_reserved_mb': 916.0}
Last epoch: {'epoch': 50, 'timestamp': '2026-10-31T14:22:26.615258', 'train_loss': 0.2942, 'train_accuracy': 92.36, 'val_loss': 1.3172, 'val_accuracy': 66.62, 'learning_rate': 1.09767694280055